# Digital Wallet — before/after load test

Compares app versions **in the same Kaggle session**: every test runs once per version, back to
back, so both versions get the same machine. (Kaggle machines differ by up to ~50 % between
sessions, so comparing runs from different sessions is not fair.)

Kaggle settings: **Accelerator = None (CPU)**, **Internet = On**. Use **Save Version → Save & Run All**;
all tests for two versions take about 2.5 hours. Remove tests from `TESTS` to go faster.

App + MySQL run on CPUs 0,2 (physical core 0), k6 on CPUs 1,3 (physical core 1).

In [ ]:
REPO_URL = 'https://github.com/LeonaRaging/SAUETDigitalWallet.git'
TESTS_FROM = 'main'                              # branch with the test files (same for every version)
BRANCHES = ['v1-baseline', 'v2-consistency']     # versions to compare (branches or tags); 1 or more
TESTS = ['smoke', 'load', 'hot', 'history', 'stress', 'uc1', 'uc2']
RATES = [100, 200, 300, 400, 500, 600, 800]      # stress levels (req/s), 2 min each

## Setup: get the code, install, start MySQL, build every version

In [ ]:
import os, subprocess, json

S   = '/tmp/tests/digitalwallet-stress-test/scripts'   # install, MySQL, build, start app
T   = '/tmp/tests/digitalwallet-stress-test/v2'        # the k6 tests
OUT = '/kaggle/working/results/' + '_vs_'.join(BRANCHES)
os.makedirs(OUT, exist_ok=True)
os.environ['JAVA_TOOL_OPTIONS'] = '-XX:ActiveProcessorCount=2'   # the app gets 2 CPUs

def sh(cmd, quiet=False):
    """Run a shell command; show its output (always on failure) and stop if it fails."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
    if r.returncode or not quiet:
        print(r.stdout + r.stderr, end='')
    if r.returncode:
        raise RuntimeError(f'failed (exit {r.returncode}): {cmd}')
    return r.stdout

def use(branch):
    """Point the setup scripts at one version's folder."""
    i = BRANCHES.index(branch)
    os.environ['SUT_DIR'] = f'/tmp/app{i}'
    os.environ['SUT_DIR_FILE'] = f'/tmp/app{i}.dir'

def clone(branch, path):
    sh(f'rm -rf {path} && git clone -q --depth 1 -b {branch} {REPO_URL} {path}')
    sh(f"find {path} -name '*.sh' -exec sed -i 's/\\r$//' {{}} +")   # Windows line endings
    return sh(f'git -C {path} rev-parse --short HEAD', quiet=True).strip()

commits = {'tests': clone(TESTS_FROM, '/tmp/tests')}
sh(f'bash {S}/00_install.sh', quiet=True)
sh(f'taskset -c 0,2 bash {S}/10_mysql.sh', quiet=True)       # MySQL on CPUs 0,2
for b in BRANCHES:
    use(b)
    commits[b] = clone(b, os.environ['SUT_DIR'])
    sh(f'bash {S}/20_build_sut.sh', quiet=True)
    print('built', b, commits[b])

machine = sh('lscpu; echo; free -h', quiet=True)
with open(f'{OUT}/machine.txt', 'w') as f:
    f.write(json.dumps(commits) + '\n\n' + machine)
print(commits); print(machine.splitlines()[0:20])

## Helpers

In [ ]:
def check_pinning():
    out = sh('taskset -cp $(pgrep -x mysqld); taskset -cp $(cat /tmp/app.pid)', quiet=True)
    assert out.count(': 0,2') == 2, f'MySQL/app not on CPUs 0,2:\n{out}'

def fresh_app(branch):
    """Empty database + this version of the app, started on CPUs 0,2."""
    use(branch)
    sh(f'bash {S}/90_reset_db.sh && taskset -c 0,2 bash {S}/30_start_app.sh', quiet=True)
    check_pinning()

def k6(branch, script, name, **settings):
    """Run one k6 test on CPUs 1,3. Saves <OUT>/<branch>/<name>.json and .txt. Returns k6's exit code."""
    os.makedirs(f'{OUT}/{branch}', exist_ok=True)
    e = ' '.join(f'-e {k}={v}' for k, v in settings.items())
    print(f'\n----- {branch}: {name} -----')
    return subprocess.run(f'set -o pipefail; taskset -c 1,3 k6 run --quiet {e} '
                          f'--summary-export {OUT}/{branch}/{name}.json {T}/{script} 2>&1 | tee {OUT}/{branch}/{name}.txt',
                          shell=True, executable='/bin/bash').returncode

def money_check(branch, name):
    """No money created or lost; saved as <name>_money.json."""
    row = sh(f'mysql -uroot -D digitalwallet_db -B -N < {T}/money_check.sql', quiet=True).split()
    total, expected, mismatched, negative = (int(float(x)) for x in row)
    result = dict(total_balance=total, expected_total=expected, wallets_not_matching_ledger=mismatched,
                  negative_wallets=negative, ok=(total == expected and mismatched == 0 and negative == 0))
    json.dump(result, open(f'{OUT}/{branch}/{name}_money.json', 'w'), indent=1)
    print('money check:', 'OK' if result['ok'] else 'VIOLATED', result)

# Every test: (script, result name, k6 settings, run the money check afterwards?)
STEPS = {
    'smoke':   [('load.js', 'smoke', dict(RATE=5, DURATION='1m', USERS=20), False)],
    'load':    [('load.js', 'load_100', dict(RATE=100, DURATION='10m'), True)],
    'hot':     [('load.js', 'hot_100', dict(RATE=100, DURATION='5m', HOT=1), True)],
    'history': [('load.js', 'history_100', dict(RATE=100, DURATION='5m', HISTORY=200), True)],
    'stress':  [('load.js', f'stress_{r}', dict(RATE=r, DURATION='2m'), False) for r in RATES],
    'uc1':     [('uc1.js', 'uc1', {}, False)],
    'uc2':     [('uc2.js', 'uc2', {}, False)],
}

## Run: each step once per version, alternating the order (A B, B A, ...) so neither version always goes first

In [ ]:
k = 0
for test in TESTS:
    for script, name, settings, check_money in STEPS[test]:
        order = BRANCHES if k % 2 == 0 else BRANCHES[::-1]
        k += 1
        for b in order:
            fresh_app(b)
            code = k6(b, script, name, **settings)
            if check_money:
                money_check(b, name)
            if test == 'smoke' and code != 0:
                raise SystemExit(f'Smoke test failed for {b}: fix this before the long tests.')
print('all done')

## Results, side by side

In [ ]:
def m(b, name, metric, field):
    try:
        return json.load(open(f'{OUT}/{b}/{name}.json'))['metrics'].get(metric, {}).get(field, 0)
    except FileNotFoundError:
        return None

def money(b, name):
    try:
        return 'OK' if json.load(open(f'{OUT}/{b}/{name}_money.json'))['ok'] else 'VIOLATED'
    except FileNotFoundError:
        return ''

runs = [('load_100', 600), ('hot_100', 300), ('history_100', 300)] + [(f'stress_{r}', 120) for r in RATES]
print(f"{'test':<12}{'version':<18}{'achieved/s':>11}{'p95 ms':>9}{'failed':>9}{'409s':>7}{'400s':>7}{'pass':>6}  money")
for name, secs in runs:
    for b in BRANCHES:
        it = m(b, name, 'iterations', 'count')
        if it is None:
            continue
        p95 = m(b, name, 'http_req_duration{scenario:load}', 'p(95)')
        fail = m(b, name, 'http_req_failed{scenario:load}', 'value')
        ok = 'yes' if p95 < 500 and fail < 0.01 else 'no'
        print(f"{name:<12}{b:<18}{it / secs:>11.1f}{p95:>9.0f}{fail:>9.2%}{m(b, name, 'conflicts_409', 'count'):>7}"
              f"{m(b, name, 'rejected_400', 'count'):>7}{ok:>6}  {money(b, name)}")
    print()

for b in BRANCHES:
    print(f"{b}: UC1 rounds debited once {m(b, 'uc1', 'rounds_debited_once', 'count')}/50, "
          f"extra debits {m(b, 'uc1', 'extra_debits', 'count')} | "
          f"UC2 both succeeded {m(b, 'uc2', 'both_succeeded', 'count')}, wrong balance {m(b, 'uc2', 'wrong_balance', 'count')}, "
          f"loser got 400: {m(b, 'uc2', 'loser_got_400', 'count')}, 409: {m(b, 'uc2', 'loser_got_409', 'count')}")

Download the zip from the Output tab and unzip it into `digitalwallet-docs/stress-test/results/`.

In [ ]:
import shutil
shutil.make_archive(OUT, 'zip', OUT)